
# 05 — Volatility Features  
## Diagnostic / Sensitivity Only

This notebook constructs pre-shock GDP growth volatility/stability features.

## Final methodological decision

GDP growth volatility/stability is **not** part of the final 5-variable POSet ordering set.

The final POSet ordering variables remain:

1. debt capacity
2. employment strength
3. R&D intensity
4. human capital
5. energy security

This notebook is retained because pre-shock volatility/stability is useful for:

- descriptive diagnostics;
- sensitivity checks;
- interpretation of resilience patterns;
- comparison with recovery outcomes.

## Important guardrail

Do **not** use post-shock years to construct pre-shock volatility features.

## Diagnostic windows

| Shock | Window |
|---|---|
| 2007 shock | 2002–2007 |
| 2019 shock | 2012–2019 |

The 2007 window starts in **2002** because the final clean acquisition pipeline starts in 2002.

## Main outputs

- `gdp_growth_stability_features_country_shock.csv`
- `volatility_feature_quality_summary.csv`
- `volatility_recovery_diagnostic_correlations.csv`
- `volatility_problem_cases.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

INPUT_DIR = PROJECT_ROOT / "Data" / "Processed" / "Comparable_Raw_Files"
RECOVERY_DIR = PROJECT_ROOT / "Data" / "Processed" / "GDP_Recovery_Dynamic_Baseline"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Volatility_Features"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "05_Volatility_Features"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Volatility_Features"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 2002
END_YEAR = 2025

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Input folder:", INPUT_DIR.resolve())
print("Recovery folder:", RECOVERY_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_184351
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Input folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files
Recovery folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\GDP_Recovery_Dynamic_Baseline
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Volatility_Features


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

STANDARDIZED_LONG_CANDIDATES = [
    INPUT_DIR / "standardized_country_year_variable_long.csv",
    INPUT_DIR / "standardized_all_variables_long.csv",
]

RECOVERY_SUMMARY_CANDIDATES = [
    RECOVERY_DIR / "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    PROJECT_ROOT / "Data" / "Diagnostics" / "03_GDP_Recovery_Dynamic_Baseline" / "country_recovery_summary_dynamic_baseline_2007_2019.csv",
]

GDP_GROWTH_VARIABLE = "gdp_growth"

VOLATILITY_WINDOWS = [
    {
        "shock_id": "2007",
        "shock_label": "Global Financial Crisis",
        "window_start": 2002,
        "window_end": 2007,
        "baseline_year": 2007,
    },
    {
        "shock_id": "2019",
        "shock_label": "COVID shock",
        "window_start": 2012,
        "window_end": 2019,
        "baseline_year": 2019,
    },
]

MIN_REQUIRED_YEARS = {
    "2007": 4,
    "2019": 5,
}

def find_first_existing_path(candidates, label, required=True):
    for path in candidates:
        if path.exists():
            print(f"Using {label}: {path}")
            return path
    if required:
        raise FileNotFoundError(
            f"Could not find {label}. Tried:\n" +
            "\n".join([f"- {p}" for p in candidates])
        )
    print(f"{label} not found. Continuing without it.")
    return None

STANDARDIZED_LONG_FILE = find_first_existing_path(
    STANDARDIZED_LONG_CANDIDATES,
    "standardized comparable long dataset",
)

RECOVERY_SUMMARY_FILE = find_first_existing_path(
    RECOVERY_SUMMARY_CANDIDATES,
    "GDP recovery summary",
    required=False,
)

print("Volatility windows:")
display(pd.DataFrame(VOLATILITY_WINDOWS))


Using standardized comparable long dataset: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Comparable_Raw_Files\standardized_country_year_variable_long.csv
Using GDP recovery summary: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\GDP_Recovery_Dynamic_Baseline\country_recovery_summary_dynamic_baseline_2007_2019.csv
Volatility windows:


,shock_id,shock_label,window_start,window_end,baseline_year
0,2007,Global Financial Crisis,2002,2007,2007
1,2019,COVID shock,2012,2019,2019


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [4]:

# ------------------------------------------------------
# LOAD GDP GROWTH DATA
# ------------------------------------------------------

df = pd.read_csv(STANDARDIZED_LONG_FILE)

rename_candidates = {
    "country": "Country",
    "iso3": "Country",
    "ISO3": "Country",
    "year": "Year",
    "Variable": "variable",
    "variable_name": "variable",
    "value": "Value",
    "obs_value": "Value",
    "OBS_VALUE": "Value",
}

for old_col, new_col in rename_candidates.items():
    if old_col in df.columns and new_col not in df.columns:
        df = df.rename(columns={old_col: new_col})

required_cols = {"Country", "Year", "variable", "Value"}
missing_cols = required_cols - set(df.columns)

if missing_cols:
    raise ValueError(
        f"Input file missing required columns: {missing_cols}\n"
        f"Selected file: {STANDARDIZED_LONG_FILE}\n"
        f"Available columns: {list(df.columns)}"
    )

df["Country"] = df["Country"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df["variable"] = df["variable"].astype(str).str.strip()

df = (
    df.dropna(subset=["Country", "Year", "variable", "Value"])
    .assign(Year=lambda d: d["Year"].astype(int))
    .query("Year >= @START_YEAR and Year <= @END_YEAR")
    .copy()
)

gdp = df[df["variable"].eq(GDP_GROWTH_VARIABLE)].copy()

if gdp.empty:
    raise ValueError(f"No GDP growth rows found for variable: {GDP_GROWTH_VARIABLE}")

invalid_country_codes = gdp[~gdp["Country"].str.match(r"^[A-Z]{3}$", na=False)].copy()
invalid_country_codes.to_csv(DIAGNOSTICS_DIR / "invalid_country_codes_in_gdp_growth.csv", index=False)

if len(invalid_country_codes) > 0:
    raise ValueError("Invalid country codes found in GDP growth data.")

gdp = (
    gdp[["Country", "Year", "Value"]]
    .rename(columns={"Value": "gdp_growth"})
    .drop_duplicates(["Country", "Year"])
    .sort_values(["Country", "Year"])
    .reset_index(drop=True)
)

print("GDP growth rows:", len(gdp))
print("Countries:", gdp["Country"].nunique())
print("Years:", gdp["Year"].min(), "-", gdp["Year"].max())
display(gdp.head())


GDP growth rows: 1043
Countries: 44
Years: 2002 - 2025


,Country,Year,gdp_growth
0,AUS,2002,3.0913
1,AUS,2003,4.2189
2,AUS,2004,3.1555
3,AUS,2005,2.7709
4,AUS,2006,3.7895


In [5]:

# ------------------------------------------------------
# BUILD PRE-SHOCK VOLATILITY / STABILITY FEATURES
# ------------------------------------------------------

feature_rows = []
window_detail_frames = []

for window in VOLATILITY_WINDOWS:
    shock_id = window["shock_id"]
    shock_label = window["shock_label"]
    start = window["window_start"]
    end = window["window_end"]
    baseline_year = window["baseline_year"]
    expected_years = list(range(start, end + 1))
    expected_year_count = len(expected_years)
    min_required = MIN_REQUIRED_YEARS[shock_id]

    subset = gdp[
        gdp["Year"].between(start, end)
    ].copy()

    subset["shock_id"] = shock_id
    subset["window_start"] = start
    subset["window_end"] = end
    window_detail_frames.append(subset)

    grouped = (
        subset
        .groupby("Country")
        .agg(
            years_available=("Year", "nunique"),
            min_year=("Year", "min"),
            max_year=("Year", "max"),
            mean_gdp_growth_pre_shock=("gdp_growth", "mean"),
            median_gdp_growth_pre_shock=("gdp_growth", "median"),
            gdp_growth_volatility_sd=("gdp_growth", "std"),
            gdp_growth_min_pre_shock=("gdp_growth", "min"),
            gdp_growth_max_pre_shock=("gdp_growth", "max"),
        )
        .reset_index()
    )

    grouped["shock_id"] = shock_id
    grouped["shock_label"] = shock_label
    grouped["baseline_year"] = baseline_year
    grouped["window_start"] = start
    grouped["window_end"] = end
    grouped["expected_year_count"] = expected_year_count
    grouped["min_required_years"] = min_required
    grouped["coverage_rate"] = grouped["years_available"] / expected_year_count
    grouped["sufficient_window_coverage"] = grouped["years_available"] >= min_required

    feature_rows.append(grouped)

gdp_growth_stability_features_country_shock = pd.concat(feature_rows, ignore_index=True)

# Score within each shock window: lower volatility = higher stability.
scored_frames = []

for shock_id, temp in gdp_growth_stability_features_country_shock.groupby("shock_id"):
    temp = temp.copy()
    eligible = temp["sufficient_window_coverage"] & temp["gdp_growth_volatility_sd"].notna()

    min_vol = temp.loc[eligible, "gdp_growth_volatility_sd"].min()
    max_vol = temp.loc[eligible, "gdp_growth_volatility_sd"].max()

    if pd.isna(min_vol) or pd.isna(max_vol) or max_vol == min_vol:
        temp["gdp_growth_stability_score_0_100"] = np.nan
    else:
        temp["gdp_growth_stability_score_0_100"] = np.where(
            eligible,
            (max_vol - temp["gdp_growth_volatility_sd"]) / (max_vol - min_vol) * 100,
            np.nan,
        )

    temp["volatility_feature_role"] = "diagnostic_or_sensitivity_only_not_final_ordering"
    scored_frames.append(temp)

gdp_growth_stability_features_country_shock = pd.concat(scored_frames, ignore_index=True)

gdp_growth_stability_features_country_shock = (
    gdp_growth_stability_features_country_shock
    .sort_values(["shock_id", "Country"])
    .reset_index(drop=True)
)

gdp_growth_pre_shock_window_detail = pd.concat(window_detail_frames, ignore_index=True)

save_table(
    gdp_growth_stability_features_country_shock,
    "gdp_growth_stability_features_country_shock.csv",
    "GDP growth stability features by country-shock",
    "Pre-shock GDP growth volatility and derived stability score. Diagnostic/sensitivity only.",
)

save_table(
    gdp_growth_pre_shock_window_detail,
    "gdp_growth_pre_shock_window_detail.csv",
    "GDP growth pre-shock window detail",
    "Country-year GDP growth values used to calculate pre-shock volatility features.",
)

display(gdp_growth_stability_features_country_shock.head())


Saved: gdp_growth_stability_features_country_shock.csv
Saved: gdp_growth_pre_shock_window_detail.csv


,Country,years_available,min_year,max_year,mean_gdp_growth_pre_shock,median_gdp_growth_pre_shock,gdp_growth_volatility_sd,gdp_growth_min_pre_shock,gdp_growth_max_pre_shock,shock_id,shock_label,baseline_year,window_start,window_end,expected_year_count,min_required_years,coverage_rate,sufficient_window_coverage,gdp_growth_stability_score_0_100,volatility_feature_role
0,AUS,6,2002,2007,3.4417,3.3897,0.5316,2.7709,4.2189,2007,Global Financial Crisis,2007,2002,2007,6,4,1.0000,True,97.9455,diagnostic_or_sensitivity_only_not_final_ordering
1,AUT,6,2002,2007,2.4260,2.4428,1.0098,1.1416,3.7752,2007,Global Financial Crisis,2007,2002,2007,6,4,1.0000,True,80.5768,diagnostic_or_sensitivity_only_not_final_ordering
2,BEL,6,2002,2007,2.4778,2.4370,1.0326,1.0380,3.6769,2007,Global Financial Crisis,2007,2002,2007,6,4,1.0000,True,79.7510,diagnostic_or_sensitivity_only_not_final_ordering
3,BRA,6,2002,2007,3.8647,3.5821,1.8429,1.1408,6.0699,2007,Global Financial Crisis,2007,2002,2007,6,4,1.0000,True,50.3209,diagnostic_or_sensitivity_only_not_final_ordering
4,CAN,6,2002,2007,2.6327,2.8186,0.5834,1.8064,3.2105,2007,Global Financial Crisis,2007,2002,2007,6,4,1.0000,True,96.0628,diagnostic_or_sensitivity_only_not_final_ordering


In [6]:

# ------------------------------------------------------
# FEATURE QUALITY SUMMARY
# ------------------------------------------------------

volatility_feature_quality_summary = (
    gdp_growth_stability_features_country_shock
    .groupby("shock_id")
    .agg(
        countries_with_any_data=("Country", "nunique"),
        countries_sufficient_coverage=("sufficient_window_coverage", "sum"),
        mean_years_available=("years_available", "mean"),
        median_years_available=("years_available", "median"),
        mean_coverage_rate=("coverage_rate", "mean"),
        mean_volatility_sd=("gdp_growth_volatility_sd", "mean"),
        median_volatility_sd=("gdp_growth_volatility_sd", "median"),
        mean_stability_score=("gdp_growth_stability_score_0_100", "mean"),
    )
    .reset_index()
)

volatility_problem_cases = gdp_growth_stability_features_country_shock[
    ~gdp_growth_stability_features_country_shock["sufficient_window_coverage"]
].copy()

save_table(
    volatility_feature_quality_summary,
    "volatility_feature_quality_summary.csv",
    "Volatility feature quality summary",
    "Coverage and quality diagnostics for pre-shock GDP volatility features.",
)

save_table(
    volatility_problem_cases,
    "volatility_problem_cases.csv",
    "Volatility problem cases",
    "Country-shock rows with insufficient pre-shock GDP growth coverage.",
)

display(volatility_feature_quality_summary)
display(volatility_problem_cases.head(100))


Saved: volatility_feature_quality_summary.csv
Saved: volatility_problem_cases.csv


,shock_id,countries_with_any_data,countries_sufficient_coverage,mean_years_available,median_years_available,mean_coverage_rate,mean_volatility_sd,median_volatility_sd,mean_stability_score
0,2007,44,44,6.0000,6.0000,1.0000,1.4756,1.4723,63.6616
1,2019,44,44,8.0000,8.0000,1.0000,1.5008,1.2124,85.2009


,Country,years_available,min_year,max_year,mean_gdp_growth_pre_shock,median_gdp_growth_pre_shock,gdp_growth_volatility_sd,gdp_growth_min_pre_shock,gdp_growth_max_pre_shock,shock_id,shock_label,baseline_year,window_start,window_end,expected_year_count,min_required_years,coverage_rate,sufficient_window_coverage,gdp_growth_stability_score_0_100,volatility_feature_role


In [7]:

# ------------------------------------------------------
# OPTIONAL DIAGNOSTIC CORRELATION WITH GDP RECOVERY
# ------------------------------------------------------
# This is only a diagnostic. It is not proof of causality and not part of POSet construction.

if RECOVERY_SUMMARY_FILE is not None and RECOVERY_SUMMARY_FILE.exists():
    recovery = pd.read_csv(RECOVERY_SUMMARY_FILE)
    recovery["Country"] = recovery["Country"].astype(str).str.strip().str.upper()

    temp = gdp_growth_stability_features_country_shock.merge(
        recovery,
        on="Country",
        how="left",
    )

    corr_rows = []

    for shock_id, group in temp.groupby("shock_id"):
        recovery_col = f"Recovery_{shock_id}"

        if recovery_col not in group.columns:
            continue

        valid = group[
            group["gdp_growth_stability_score_0_100"].notna()
            & pd.to_numeric(group[recovery_col], errors="coerce").notna()
        ].copy()

        valid[recovery_col] = pd.to_numeric(valid[recovery_col], errors="coerce")

        if len(valid) >= 3:
            corr_rows.append({
                "shock_id": shock_id,
                "n_countries": len(valid),
                "corr_stability_score_vs_recovery_years": valid["gdp_growth_stability_score_0_100"].corr(valid[recovery_col]),
                "corr_volatility_sd_vs_recovery_years": valid["gdp_growth_volatility_sd"].corr(valid[recovery_col]),
                "note": "Diagnostic only. Recovery is not used as a POSet ordering variable.",
            })
        else:
            corr_rows.append({
                "shock_id": shock_id,
                "n_countries": len(valid),
                "corr_stability_score_vs_recovery_years": np.nan,
                "corr_volatility_sd_vs_recovery_years": np.nan,
                "note": "Too few valid rows for correlation.",
            })

    volatility_recovery_diagnostic_correlations = pd.DataFrame(corr_rows)

else:
    volatility_recovery_diagnostic_correlations = pd.DataFrame([
        {
            "shock_id": "not_run",
            "n_countries": 0,
            "corr_stability_score_vs_recovery_years": np.nan,
            "corr_volatility_sd_vs_recovery_years": np.nan,
            "note": "Recovery summary not found. Run Notebook 03 first to create diagnostic correlations.",
        }
    ])

save_table(
    volatility_recovery_diagnostic_correlations,
    "volatility_recovery_diagnostic_correlations.csv",
    "Volatility recovery diagnostic correlations",
    "Diagnostic correlations between pre-shock stability and GDP recovery outcome.",
)

display(volatility_recovery_diagnostic_correlations)


Saved: volatility_recovery_diagnostic_correlations.csv


,shock_id,n_countries,corr_stability_score_vs_recovery_years,corr_volatility_sd_vs_recovery_years,note
0,2007,44,0.0343,-0.0343,Diagnostic only. Recovery is not used as a POS...
1,2019,43,0.1265,-0.1265,Diagnostic only. Recovery is not used as a POS...


In [8]:

# ------------------------------------------------------
# REPORT-READY METHOD NOTE
# ------------------------------------------------------

volatility_methodological_note = pd.DataFrame([
    {
        "topic": "Volatility role",
        "report_text": (
            "GDP growth stability was computed from pre-shock windows as a diagnostic and possible "
            "sensitivity feature. It was not included among the five final POSet ordering variables."
        ),
    },
    {
        "topic": "Window choice",
        "report_text": (
            "The 2007 shock used a 2002–2007 pre-shock diagnostic window, while the 2019 shock used "
            "a 2012–2019 pre-shock diagnostic window. Post-shock years were excluded from the volatility calculation."
        ),
    },
    {
        "topic": "Direction",
        "report_text": (
            "Lower GDP growth volatility was interpreted as greater pre-shock stability. The derived "
            "0–100 stability score was therefore direction-aligned so that higher values indicate greater stability."
        ),
    },
])

save_table(
    volatility_methodological_note,
    "volatility_methodological_note.csv",
    "Volatility methodological note",
    "Report-ready wording for volatility feature role and construction.",
)

display(volatility_methodological_note)


Saved: volatility_methodological_note.csv


,topic,report_text
0,Volatility role,GDP growth stability was computed from pre-sho...
1,Window choice,The 2007 shock used a 2002–2007 pre-shock diag...
2,Direction,Lower GDP growth volatility was interpreted as...


In [9]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "05_Volatility_Features_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    volatility_feature_quality_summary.to_excel(writer, sheet_name="quality_summary", index=False)
    gdp_growth_stability_features_country_shock.to_excel(writer, sheet_name="stability_features", index=False)
    volatility_problem_cases.to_excel(writer, sheet_name="problem_cases", index=False)
    volatility_recovery_diagnostic_correlations.to_excel(writer, sheet_name="recovery_correlations", index=False)
    volatility_methodological_note.to_excel(writer, sheet_name="method_note", index=False)
    gdp_growth_pre_shock_window_detail.to_excel(writer, sheet_name="window_detail", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\05_Volatility_Features_Audit.xlsx


In [10]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "volatility_features_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "volatility_features_table_inventory.csv", index=False)

expected_outputs = [
    "gdp_growth_stability_features_country_shock.csv",
    "gdp_growth_pre_shock_window_detail.csv",
    "volatility_feature_quality_summary.csv",
    "volatility_problem_cases.csv",
    "volatility_recovery_diagnostic_correlations.csv",
    "volatility_methodological_note.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("05 VOLATILITY FEATURES COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 05_Volatility_Features_Audit.xlsx")
print("- gdp_growth_stability_features_country_shock.csv")
print("- volatility_feature_quality_summary.csv")

print("\nNext notebook:")
print("06_Master_Dataset_Build_2002_5Var.ipynb")


05 VOLATILITY FEATURES COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,gdp_growth_stability_features_country_shock.csv,True,True,True
1,gdp_growth_pre_shock_window_detail.csv,True,True,True
2,volatility_feature_quality_summary.csv,True,True,True
3,volatility_problem_cases.csv,True,True,True
4,volatility_recovery_diagnostic_correlations.csv,True,True,True
5,volatility_methodological_note.csv,True,True,True



Key outputs to inspect/send:
- 05_Volatility_Features_Audit.xlsx
- gdp_growth_stability_features_country_shock.csv
- volatility_feature_quality_summary.csv

Next notebook:
06_Master_Dataset_Build_2002_5Var.ipynb
